# SupportIQ – Customer Support Intelligence Platform (Databricks)

Adaptado do kernel Kaggle de Hanumathureddygari para o workspace Databricks do challenge 002.

1. Rode o **LAB 01** (`01_LAB_importando_dados/lab01_carga_csv`) para criar `workspace.supportiq.customer_support_tickets`.
2. Widgets: `catalog_name`, `schema_name`, `data_path` (fallback CSV em `dados/` se a tabela Delta ainda nao existir).
3. Colunas = header do CSV (17, com espacos). Dataset 2 (`Topic_group`) **nao esta** neste pacote — nao inventar.
4. `Time to Resolution` e datetime; horas = TTR − FRT, so Closed.
5. Cluster ou serverless → **Run All**.


<div style="padding:20px;color:white;margin:0;font-size:140%;text-align:center;display:fill;border-radius:18px;background:linear-gradient(90deg,#2563EB,#7C3AED);overflow:hidden;font-weight:800;box-shadow:0 6px 18px rgba(37,99,235,0.3)">
SupportIQ – Customer Support Intelligence Platform
</div>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATE_COLUMNS = {"Date of Purchase"}
TIMESTAMP_COLUMNS = {"First Response Time", "Time to Resolution"}


def widget_or_default(name, default):
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def normalize_frame(frame):
    for col in frame.columns:
        if col in DATE_COLUMNS or col in TIMESTAMP_COLUMNS:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    if "Customer Satisfaction Rating" in frame.columns:
        frame["Customer Satisfaction Rating"] = pd.to_numeric(
            frame["Customer Satisfaction Rating"], errors="coerce"
        )
    return frame


def resolve_data_path(explicit):
    marker = "customer_support_tickets.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-process-002-support" in parts:
            idx = parts.index("notebook-process-002-support")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend([here / "dados", here.parent / "dados", here.parent.parent / "dados"])
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError("Rode o LAB 01 ou defina data_path para a pasta dados/.")


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "supportiq")
data_path_widget = widget_or_default("data_path", "")

try:
    df = spark.table(f"{catalog_name}.{schema_name}.customer_support_tickets").toPandas()
    print(f"loaded Delta {catalog_name}.{schema_name}.customer_support_tickets")
except Exception as exc:
    print("Delta indisponivel, lendo CSV:", exc)
    data_path = resolve_data_path(data_path_widget)
    df = pd.read_csv(f"{data_path}/customer_support_tickets.csv")
    print(f"loaded CSV from {data_path}")

df = normalize_frame(df)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


In [ ]:
print(df.dtypes)
df.head()

In [ ]:
assert list(df.columns) == [
    "Ticket ID", "Customer Name", "Customer Email", "Customer Age",
    "Customer Gender", "Product Purchased", "Date of Purchase",
    "Ticket Type", "Ticket Subject", "Ticket Description",
    "Ticket Status", "Resolution", "Ticket Priority", "Ticket Channel",
    "First Response Time", "Time to Resolution",
    "Customer Satisfaction Rating",
], df.columns.tolist()
print("contract ok", df.shape)

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
status_check = df.groupby("Ticket Status").agg(
    total_tickets=("Ticket ID", "count"),
    missing_resolution=("Resolution", lambda x: x.isna().sum()),
    missing_resolution_time=("Time to Resolution", lambda x: x.isna().sum()),
    missing_satisfaction=("Customer Satisfaction Rating", lambda x: x.isna().sum())
)

print(status_check)

In [ ]:
closed_df = df[df["Ticket Status"] == "Closed"].copy()

print("Closed tickets:", len(closed_df))
print("Shape:", closed_df.shape)

print("\nMissing values in closed tickets:")
print(closed_df.isnull().sum())

In [ ]:
# Convert date columns to datetime
closed_df["Date of Purchase"] = pd.to_datetime(closed_df["Date of Purchase"])
closed_df["First Response Time"] = pd.to_datetime(closed_df["First Response Time"])
closed_df["Time to Resolution"] = pd.to_datetime(closed_df["Time to Resolution"])

# Create useful date features
closed_df["purchase_year"] = closed_df["Date of Purchase"].dt.year
closed_df["purchase_month"] = closed_df["Date of Purchase"].dt.month

print(closed_df[
    ["Date of Purchase", "First Response Time", "Time to Resolution",
     "purchase_year", "purchase_month"]
].head())

In [ ]:
closed_df["resolution_hours"] = (
    closed_df["Time to Resolution"] - closed_df["First Response Time"]
).dt.total_seconds() / 3600

print("Resolution hours:")
print(closed_df["resolution_hours"].describe())

print("\nNegative resolution times:")
print((closed_df["resolution_hours"] < 0).sum())

print("\nSample:")
print(
    closed_df[
        ["First Response Time", "Time to Resolution", "resolution_hours"]
    ].head(10)
)

In [ ]:
closed_df["response_hour"] = closed_df["First Response Time"].dt.hour
closed_df["resolution_hour"] = closed_df["Time to Resolution"].dt.hour

print("Response hour distribution:")
print(closed_df["response_hour"].value_counts().sort_index())

print("\nResolution hour distribution:")
print(closed_df["resolution_hour"].value_counts().sort_index())

print("\nResolution date distribution:")
print(closed_df["Time to Resolution"].dt.date.value_counts().sort_index())

In [ ]:
# Calculate resolution time in hours
closed_df["resolution_hours"] = (
    closed_df["Time to Resolution"] - closed_df["First Response Time"]
).dt.total_seconds() / 3600

# If resolution time is negative, assume it crossed midnight
closed_df.loc[
    closed_df["resolution_hours"] < 0,
    "resolution_hours"
] += 24

print(closed_df["resolution_hours"].describe())

print("\nNegative values after correction:")
print((closed_df["resolution_hours"] < 0).sum())

In [ ]:
print("\nResolution time examples:")
print(
    closed_df[
        ["First Response Time", "Time to Resolution", "resolution_hours"]
    ].head(10)
)

In [ ]:
# Create simple features from ticket information

closed_df["subject_length"] = closed_df["Ticket Subject"].str.len()
closed_df["description_length"] = closed_df["Ticket Description"].str.len()

closed_df["purchase_year"] = closed_df["Date of Purchase"].dt.year
closed_df["purchase_month"] = closed_df["Date of Purchase"].dt.month

closed_df["response_hour"] = closed_df["First Response Time"].dt.hour

print("New features created:")
print(
    closed_df[
        [
            "subject_length",
            "description_length",
            "purchase_year",
            "purchase_month",
            "response_hour",
            "resolution_hours",
            "Customer Satisfaction Rating"
        ]
    ].head()
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features used for satisfaction prediction
features = [
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Ticket Type",
    "Ticket Subject",
    "Ticket Description",
    "Ticket Priority",
    "Ticket Channel",
    "subject_length",
    "description_length",
    "purchase_year",
    "purchase_month",
    "response_hour"
]

X = closed_df[features]
y = closed_df["Customer Satisfaction Rating"].astype(int)

# Separate numeric and categorical features
numeric_features = [
    "Customer Age",
    "subject_length",
    "description_length",
    "purchase_year",
    "purchase_month",
    "response_hour"
]

categorical_features = [
    "Customer Gender",
    "Product Purchased",
    "Ticket Type",
    "Ticket Subject",
    "Ticket Description",
    "Ticket Priority",
    "Ticket Channel"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

# Model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=42
    ))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression

# Combine subject and description into one text field
closed_df["ticket_text"] = (
    closed_df["Ticket Subject"].fillna("") + " " +
    closed_df["Ticket Description"].fillna("")
)

X_text = closed_df["ticket_text"]
y = closed_df["Customer Satisfaction Rating"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert ticket text into numerical features
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train classifier
satisfaction_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

satisfaction_model.fit(X_train_tfidf, y_train)

# Predict
y_pred = satisfaction_model.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
print("Satisfaction rating distribution:")
print(
    closed_df["Customer Satisfaction Rating"]
    .value_counts()
    .sort_index()
)

print("\nAverage satisfaction by priority:")
print(
    closed_df.groupby("Ticket Priority")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

print("\nAverage satisfaction by ticket type:")
print(
    closed_df.groupby("Ticket Type")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

In [ ]:
print("Average satisfaction by channel:")
print(
    closed_df.groupby("Ticket Channel")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

print("\nAverage satisfaction by product:")
print(
    closed_df.groupby("Product Purchased")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

print("\nAverage satisfaction by customer gender:")
print(
    closed_df.groupby("Customer Gender")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

print("\nSatisfaction correlation with customer age:")
print(
    closed_df[
        ["Customer Age", "Customer Satisfaction Rating"]
    ].corr()
)

In [ ]:
print("Average satisfaction by channel:")
print(
    closed_df.groupby("Ticket Channel")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

print("\nAverage satisfaction by product:")
print(
    closed_df.groupby("Product Purchased")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

print("\nAverage satisfaction by customer gender:")
print(
    closed_df.groupby("Customer Gender")[
        "Customer Satisfaction Rating"
    ].mean().sort_values()
)

print("\nSatisfaction correlation with customer age:")
print(
    closed_df[
        ["Customer Age", "Customer Satisfaction Rating"]
    ].corr()
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Features with useful structured information
structured_features = [
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel",
    "purchase_year",
    "purchase_month",
    "response_hour",
    "subject_length",
    "description_length"
]

X = closed_df[structured_features]
y = closed_df["Customer Satisfaction Rating"].astype(int)

numeric_features = [
    "Customer Age",
    "purchase_year",
    "purchase_month",
    "response_hour",
    "subject_length",
    "description_length"
]

categorical_features = [
    "Customer Gender",
    "Product Purchased",
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

structured_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

structured_model.fit(X_train, y_train)

y_pred = structured_model.predict(X_test)

print("Structured model accuracy:",
      round(accuracy_score(y_test, y_pred), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
print("Resolution time statistics:")
print(closed_df["resolution_hours"].describe())

print("\nResolution time percentiles:")
print(
    closed_df["resolution_hours"].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Features for resolution-time prediction
resolution_features = [
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel",
    "purchase_year",
    "purchase_month",
    "response_hour",
    "subject_length",
    "description_length"
]

X = closed_df[resolution_features]
y = closed_df["resolution_hours"]

numeric_features = [
    "Customer Age",
    "purchase_year",
    "purchase_month",
    "response_hour",
    "subject_length",
    "description_length"
]

categorical_features = [
    "Customer Gender",
    "Product Purchased",
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

resolution_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

resolution_model.fit(X_train, y_train)

y_pred = resolution_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Resolution Time Model")
print("---------------------")
print("MAE:", round(mae, 2), "hours")
print("RMSE:", round(rmse, 2), "hours")
print("R2 Score:", round(r2, 4))

In [ ]:
print("Average resolution time by priority:")
print(
    closed_df.groupby("Ticket Priority")["resolution_hours"]
    .mean()
    .sort_values()
)

print("\nAverage resolution time by ticket type:")
print(
    closed_df.groupby("Ticket Type")["resolution_hours"]
    .mean()
    .sort_values()
)

print("\nAverage resolution time by channel:")
print(
    closed_df.groupby("Ticket Channel")["resolution_hours"]
    .mean()
    .sort_values()
)

print("\nCorrelation with numeric features:")
print(
    closed_df[
        [
            "Customer Age",
            "purchase_year",
            "purchase_month",
            "response_hour",
            "subject_length",
            "description_length",
            "resolution_hours"
        ]
    ].corr()["resolution_hours"].sort_values()
)

In [ ]:
customer_data = closed_df.groupby(
    ["Customer Name", "Customer Email"]
).agg(
    customer_age=("Customer Age", "first"),
    ticket_count=("Ticket ID", "count"),
    avg_satisfaction=("Customer Satisfaction Rating", "mean"),
    avg_resolution_hours=("resolution_hours", "mean")
).reset_index()

print("Customer dataset shape:", customer_data.shape)

print("\nSample:")
print(customer_data.head())

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Select features for segmentation
features = customer_data[
    [
        "customer_age",
        "ticket_count",
        "avg_satisfaction",
        "avg_resolution_hours"
    ]
]

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Test different numbers of clusters
silhouette_scores = {}

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_features)

    score = silhouette_score(scaled_features, labels)
    silhouette_scores[k] = score

print("Silhouette Scores:")

for k, score in silhouette_scores.items():
    print(f"k={k}: {score:.3f}")

In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)

customer_data["cluster"] = kmeans.fit_predict(scaled_features)

print("Cluster counts:")
print(customer_data["cluster"].value_counts().sort_index())

In [ ]:
cluster_summary = customer_data.groupby("cluster").agg(
    customers=("Customer Email", "count"),
    avg_age=("customer_age", "mean"),
    avg_ticket_count=("ticket_count", "mean"),
    avg_satisfaction=("avg_satisfaction", "mean"),
    avg_resolution_hours=("avg_resolution_hours", "mean")
).round(2)

print(cluster_summary)

In [ ]:
# Use meaningful customer features
features = customer_data[
    [
        "customer_age",
        "avg_satisfaction",
        "avg_resolution_hours"
    ]
]

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Create 6 customer segments
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)

customer_data["cluster"] = kmeans.fit_predict(scaled_features)

# Show the new cluster summary
cluster_summary = customer_data.groupby("cluster").agg(
    customers=("Customer Email", "count"),
    avg_age=("customer_age", "mean"),
    avg_satisfaction=("avg_satisfaction", "mean"),
    avg_resolution_hours=("avg_resolution_hours", "mean")
).round(2)

print(cluster_summary)

## Customer Segmentation

Customer segmentation was performed using K-Means clustering based on customer age, average satisfaction rating, and average ticket resolution time.

The features were standardized before clustering so that each feature contributed fairly to the clustering process.

Six customer segments were created to identify different customer behavior and support experience patterns.

In [ ]:
# Give the clusters meaningful business names

cluster_names = {
    0: "Young - Satisfied but Slow",
    1: "Older - At Risk and Slow",
    2: "Older - Satisfied but Slow",
    3: "Young - At Risk and Fast",
    4: "Middle Age - Highly Satisfied and Fast",
    5: "Older - At Risk and Fast"
}

customer_data["segment"] = customer_data["cluster"].map(cluster_names)

# Show final segment summary
segment_summary = customer_data.groupby("segment").agg(
    customers=("Customer Email", "count"),
    avg_age=("customer_age", "mean"),
    avg_satisfaction=("avg_satisfaction", "mean"),
    avg_resolution_hours=("avg_resolution_hours", "mean")
).round(2)

print(segment_summary)

## Customer Segment Insights

The K-Means model identified six distinct customer segments based on age, satisfaction, and resolution time.

### Key Findings

- **Middle Age - Highly Satisfied and Fast:** Highest satisfaction at 4.44 with the fastest average resolution time of 5.04 hours.
- **Older - At Risk and Slow:** Lowest satisfaction at 1.52 and the slowest resolution time at 18.81 hours. This segment should be a priority for customer support improvement.
- **Older - Satisfied but Slow:** Customers have high satisfaction (4.11) despite relatively slow resolution of 16.02 hours.
- **Young - At Risk and Fast:** Fast resolution of 6.93 hours but low satisfaction of 1.84, suggesting that resolution speed alone does not guarantee customer satisfaction.
- **Older - At Risk and Fast:** Resolution is fast at 5.64 hours, but satisfaction remains low at 2.16.
- **Young - Satisfied but Slow:** Satisfaction is relatively positive at 3.77, although average resolution time is high at 17.81 hours.

These segments can help support teams prioritize at-risk customers and identify customer groups with different service experience patterns.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

for segment in customer_data["segment"].unique():
    segment_data = customer_data[customer_data["segment"] == segment]

    plt.scatter(
        segment_data["avg_resolution_hours"],
        segment_data["avg_satisfaction"],
        label=segment,
        alpha=0.6
    )

plt.xlabel("Average Resolution Time (Hours)")
plt.ylabel("Average Satisfaction Rating")
plt.title("Customer Segments by Satisfaction and Resolution Time")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)

plt.tight_layout()
plt.show()

## Final Customer Segmentation Results

The K-Means clustering model successfully grouped 2,769 closed support tickets into six customer segments.

The segmentation uses standardized customer age, average satisfaction rating, and average resolution time. These segments provide actionable insights into customer experience and can help support teams identify at-risk customers and prioritize service improvements.

In [ ]:
import joblib

# Save the trained scaler and clustering model
joblib.dump(scaler, "customer_segmentation_scaler.pkl")
joblib.dump(kmeans, "customer_segmentation_model.pkl")

# Save customer segmentation results
customer_data.to_csv("customer_segments.csv", index=False)

# Save the segment summary
segment_summary.to_csv("segment_summary.csv")

print("Segmentation model saved successfully.")
print("Customer segments saved successfully.")
print("Segment summary saved successfully.")

## Customer Satisfaction Prediction

The original satisfaction rating contains five classes, and the multiclass model showed limited predictive performance.

For a more actionable business use case, satisfaction is converted into two categories:

- **Low Satisfaction:** Rating 1 or 2
- **Satisfied:** Rating 3, 4, or 5

This allows the model to focus on identifying customers who may require additional support attention.

In [ ]:
# Create a binary satisfaction target

closed_df["satisfaction_status"] = closed_df[
    "Customer Satisfaction Rating"
].apply(
    lambda rating: "Low Satisfaction" if rating <= 2 else "Satisfied"
)

print("Satisfaction distribution:")
print(closed_df["satisfaction_status"].value_counts())

print("\nPercentage:")
print(
    closed_df["satisfaction_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

## Satisfaction Risk Prediction Model

A Random Forest classifier is used to predict whether a support ticket is likely to have low customer satisfaction.

The model uses customer, ticket, and support-channel information available before the satisfaction rating is known.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Select input features
X = closed_df[
    [
        "Customer Age",
        "Customer Gender",
        "Product Purchased",
        "Ticket Type",
        "Ticket Priority",
        "Ticket Channel",
        "purchase_year",
        "purchase_month",
        "response_hour",
        "subject_length",
        "description_length"
    ]
]

y = closed_df["satisfaction_status"]

# Separate categorical and numerical features
categorical_features = [
    "Customer Gender",
    "Product Purchased",
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel"
]

numerical_features = [
    "Customer Age",
    "purchase_year",
    "purchase_month",
    "response_hour",
    "subject_length",
    "description_length"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

# Build the model
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Improving Satisfaction Risk Detection

The first classification model achieved 59.39% accuracy but detected only 15% of low-satisfaction tickets.

To reduce the effect of class imbalance, a class-balanced Random Forest model is tested. The goal is to improve detection of customers with low satisfaction rather than optimizing accuracy alone.

In [ ]:
# Create a class-balanced Random Forest model

balanced_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

# Train the balanced model
balanced_model.fit(X_train, y_train)

# Make predictions
balanced_pred = balanced_model.predict(X_test)

# Evaluate
balanced_accuracy = accuracy_score(y_test, balanced_pred)

print(f"Accuracy: {balanced_accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, balanced_pred))

## Satisfaction Model Result

The class-balanced Random Forest achieved 59.75% accuracy. However, its recall for low-satisfaction tickets remained low at 14%.

This indicates that the available customer and ticket features have limited predictive power for satisfaction in this dataset.

The model is therefore treated as an experimental prediction component, while customer segmentation and support analytics provide the primary actionable insights.

In [ ]:
# Save the satisfaction prediction model

joblib.dump(
    balanced_model,
    "satisfaction_prediction_model.pkl"
)

print("Satisfaction model saved successfully.")

## Resolution Time Analysis

Resolution time is analyzed to identify patterns across ticket priority, ticket type, and support channel.

The analysis uses closed tickets because only closed tickets contain a completed resolution time.

The resulting insights can help support teams understand which types of tickets take longer to resolve.

In [ ]:
# Resolution time by ticket priority
priority_resolution = closed_df.groupby(
    "Ticket Priority"
)["resolution_hours"].mean().sort_values()

print("Average Resolution Time by Priority:")
print(priority_resolution.round(2))

# Resolution time by ticket type
type_resolution = closed_df.groupby(
    "Ticket Type"
)["resolution_hours"].mean().sort_values()

print("\nAverage Resolution Time by Ticket Type:")
print(type_resolution.round(2))

# Resolution time by support channel
channel_resolution = closed_df.groupby(
    "Ticket Channel"
)["resolution_hours"].mean().sort_values()

print("\nAverage Resolution Time by Channel:")
print(channel_resolution.round(2))

## Resolution Time Insights

The average resolution time across the available ticket categories is approximately 11–12 hours.

### Key Findings

- **Priority:** Critical tickets have the shortest average resolution time at 11.44 hours, while Low priority tickets average 11.94 hours.
- **Ticket Type:** Product inquiries take the longest to resolve at 12.18 hours, while Technical issues take 11.45 hours.
- **Channel:** Phone tickets have the longest average resolution time at 12.09 hours. Email and Chat are the fastest at approximately 11.55 hours.
- The differences between groups are relatively small, indicating that the available ticket attributes have limited influence on resolution time in this dataset.

These findings can help support teams monitor resolution patterns across ticket categories and channels.

In [ ]:
# Save resolution-time analysis results

priority_resolution.to_csv("resolution_by_priority.csv")
type_resolution.to_csv("resolution_by_type.csv")
channel_resolution.to_csv("resolution_by_channel.csv")

print("Resolution analysis files saved successfully.")

## Dashboard Metrics

The following metrics are prepared for the SupportIQ dashboard.

The dashboard will present customer segmentation, satisfaction insights, and ticket resolution patterns in a simple business-oriented interface.

In [ ]:
# Create summary metrics for the dashboard

dashboard_metrics = {
    "total_tickets": len(df),
    "closed_tickets": len(closed_df),
    "open_tickets": len(df[df["Ticket Status"] == "Open"]),
    "pending_tickets": len(
        df[df["Ticket Status"] == "Pending Customer Response"]
    ),
    "average_satisfaction": round(
        closed_df["Customer Satisfaction Rating"].mean(), 2
    ),
    "average_resolution_hours": round(
        closed_df["resolution_hours"].mean(), 2
    ),
    "low_satisfaction_tickets": int(
        (closed_df["Customer Satisfaction Rating"] <= 2).sum()
    ),
    "satisfied_tickets": int(
        (closed_df["Customer Satisfaction Rating"] >= 3).sum()
    ),
    "customer_segments": customer_data["segment"].nunique()
}

dashboard_metrics_df = pd.DataFrame(
    list(dashboard_metrics.items()),
    columns=["metric", "value"]
)

dashboard_metrics_df.to_csv(
    "dashboard_metrics.csv",
    index=False
)

print(dashboard_metrics_df)
print("\nDashboard metrics saved successfully.")

## Dashboard Metrics

The SupportIQ dashboard summarizes the customer support dataset and presents key operational indicators.

The dataset contains 8,469 tickets, including 2,769 closed tickets, 2,819 open tickets, and 2,881 tickets pending customer response.

Among closed tickets, the average customer satisfaction rating is 2.99 and the average resolution time is 11.77 hours. A total of 1,102 tickets received a satisfaction rating of 1 or 2, indicating low satisfaction.

Six customer segments were identified using K-Means clustering.

In [ ]:
# Prepare customer segment data for the dashboard

segment_dashboard = segment_summary.reset_index()

segment_dashboard.columns = [
    "segment",
    "customers",
    "avg_age",
    "avg_satisfaction",
    "avg_resolution_hours"
]

segment_dashboard.to_json(
    "segment_dashboard.json",
    orient="records",
    indent=2
)

print("Segment dashboard data saved successfully.")
print(segment_dashboard)

## Frontend Data Preparation

The processed machine learning results are converted into JSON files so that they can be displayed directly in the SupportIQ web dashboard.

The dashboard will use these files to display satisfaction patterns, resolution-time patterns, and customer segment information.

In [ ]:
# Prepare resolution-time data for the dashboard

priority_dashboard = priority_resolution.reset_index()
priority_dashboard.columns = ["priority", "avg_resolution_hours"]

type_dashboard = type_resolution.reset_index()
type_dashboard.columns = ["ticket_type", "avg_resolution_hours"]

channel_dashboard = channel_resolution.reset_index()
channel_dashboard.columns = ["channel", "avg_resolution_hours"]

# Save as JSON
priority_dashboard.to_json(
    "resolution_by_priority.json",
    orient="records",
    indent=2
)

type_dashboard.to_json(
    "resolution_by_type.json",
    orient="records",
    indent=2
)

channel_dashboard.to_json(
    "resolution_by_channel.json",
    orient="records",
    indent=2
)

print("Resolution dashboard data saved successfully.")

## Satisfaction Dashboard Data

Customer satisfaction results are prepared for visualization in the SupportIQ dashboard.

The dashboard will show the distribution between satisfied and low-satisfaction tickets and help highlight potential customer experience concerns.

In [ ]:
# Prepare satisfaction data for the dashboard

satisfaction_dashboard = (
    closed_df["satisfaction_status"]
    .value_counts()
    .reset_index()
)

satisfaction_dashboard.columns = [
    "satisfaction_status",
    "ticket_count"
]

# Calculate percentage
satisfaction_dashboard["percentage"] = (
    satisfaction_dashboard["ticket_count"]
    / satisfaction_dashboard["ticket_count"].sum()
    * 100
).round(2)

# Save as JSON
satisfaction_dashboard.to_json(
    "satisfaction_dashboard.json",
    orient="records",
    indent=2
)

print("Satisfaction dashboard data saved successfully.")
print(satisfaction_dashboard)

## Final Project Outputs

The processed data, trained machine learning models, and dashboard datasets are now prepared.

These files will be used to build the SupportIQ frontend and document the complete end-to-end solution.

In [ ]:
import os

project_files = [
    "customer_segmentation_model.pkl",
    "customer_segmentation_scaler.pkl",
    "satisfaction_prediction_model.pkl",
    "customer_segments.csv",
    "segment_summary.csv",
    "resolution_by_priority.csv",
    "resolution_by_type.csv",
    "resolution_by_channel.csv",
    "dashboard_metrics.csv",
    "segment_dashboard.json",
    "resolution_by_priority.json",
    "resolution_by_type.json",
    "resolution_by_channel.json",
    "satisfaction_dashboard.json"
]

print("Final project files:")
print()

for file in project_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024
        print(f"✓ {file} ({size:.1f} KB)")
    else:
        print(f"✗ {file} - MISSING")

# SupportIQ — ML Project Summary

The SupportIQ project provides an end-to-end customer support analytics solution using the Customer Support Ticket Dataset.

### Machine Learning Components

1. **Customer Segmentation**
   - Algorithm: K-Means Clustering
   - Features: Customer Age, Average Satisfaction, Average Resolution Time
   - Number of segments: 6
   - Silhouette Score: 0.295

2. **Customer Satisfaction Risk Analysis**
   - Algorithm: Random Forest Classification
   - Target: Low Satisfaction vs Satisfied
   - Accuracy: 59.75%
   - The model showed limited predictive power for low-satisfaction tickets.

3. **Resolution Time Analysis**
   - Average resolution time: 11.77 hours
   - Analysis performed across priority, ticket type, and support channel.
   - The available features showed limited predictive power for resolution time.

### Business Value

The solution identifies customer groups with different support experiences, highlights low-satisfaction customers, and provides operational insights into ticket resolution patterns.